# Ensemble com Selecao Automatica de Metodos — Datasets Sinteticos Conhecidos (Toy A / Toy B)

**Versao 3 — ativa a busca combinatoria de subconjuntos ja existente no framework**, em vez de
forcar sempre os 8 metodos ("8-de-8", padrao herdado do notebook Traffic). A funcao
`select_robust_ensemble_combination` ja aceita `min_methods`/`max_methods` e testa todas as
combinacoes de metodos nesse intervalo, escolhendo a de maior `performance_score` — um escore
interno calculado so a partir de sinais cegos (estabilidade entre reamostragens bootstrap,
confianca, probabilidade media da aresta, densidade), **sem nunca consultar o ground truth**:

```
performance_score = 0.40*estabilidade_media + 0.20*confianca_media + 0.15*prob_aresta_media
                   + 0.15*taxa_arestas_estaveis + 0.10*(1 - sqrt(densidade))
```

Este notebook compara tres estrategias: `ENSEMBLE` (soft voting fixo com os 8 metodos, igual as
versoes anteriores), `ENSEMBLE_AUTO` (soft voting sobre o subconjunto de metodos escolhido
automaticamente pelo `performance_score`) e os 8 metodos avulsos — nas mesmas 10 replicas
independentes por dataset (seeds 1001-1010) e no mesmo protocolo estatistico pareado (Wilcoxon +
Holm + IC 95% + taxa de vitoria) das versoes anteriores. `MAX_LAG=2` e
`RANKED_SELECTION_MAX_PAIR_DENSITY=0.40` continuam nos valores "cegos" (nao calibrados neste grafo
toy) herdados do notebook Traffic.

**Custo:** ~325s/replica, medido diretamente. Testamos paralelizar a busca (`ThreadPoolExecutor`,
igual ao padrao ja usado internamente para os 8 metodos) e tambem aumentar `parallel_jobs` de 4
para 12 nucleos — nenhum dos dois ajudou (325s -> ~390s com a busca paralela; 127.7s -> 127.1s na
fase de base+bootstrap com mais threads). O gargalo real nao e falta de nucleos: e o tempo
sequencial intrinseco dos metodos mais lentos (NeuralGrangercMLP, LPCMCI) e overhead do GIL do
Python nas ~250 avaliacoes de subconjunto, que fazem muitas operacoes pequenas de pandas/sklearn
em vez de poucas operacoes grandes que se beneficiariam de mais threads. Por isso o laco de busca
permanece sequencial. `precompute_runs=True` garante que cada metodo (base + bootstrap) roda uma
unica vez por replica; a busca so reaproveita esses resultados, nunca reexecuta os metodos.

**Nenhum metodo, nem qualquer das duas montagens do ensemble, recebe o ground truth em nenhum
momento da execucao** — ele so aparece nas celulas de avaliacao, depois que as previsoes ja foram
feitas. As 10 replicas usam a mesma estrutura causal (mesmos coeficientes do gerador), variando
apenas a realizacao de ruido.

**Limite da conclusao:** mesmo com o criterio estatistico atendido, o resultado fica restrito ao
grafo toy_a (ou toy_b) especifico — nao generaliza para outros grafos.

In [1]:
from pathlib import Path
import json
import os
import pickle
import time

import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display

from causal_discovery import (
    CausalPreprocessor,
    add_precision_consensus_selection,
    build_complete_undirected_pair_scores,
    compute_paired_superiority_statistics,
    compute_ranked_undirected_skeleton_metrics,
    compute_undirected_skeleton_metrics,
    get_registered_method_kwargs,
    get_registered_method_weights,
    get_registered_methods,
    load_time_series_dataset,
)
from causal_discovery.ensemble_selection import (
    add_ranked_structure_selection,
    select_robust_ensemble_combination,
)

## 1. Configuracao

`MAX_LAG = 2` e `RANKED_SELECTION_MAX_PAIR_DENSITY = 0.40` continuam nos mesmos valores "cegos" do
notebook Traffic. Novidade desta versao: `MIN_METHODS = 2` e `MAX_METHODS = 8` habilitam a busca
combinatoria de subconjuntos (em vez de `min_methods=max_methods=8`, que forcava sempre todos os
metodos). `N_BOOTSTRAP = 10` por replica. Dez replicas independentes por dataset (seeds
1001-1010 — as mesmas da versao 2) permitem aplicar o mesmo criterio confirmatorio do Traffic: IC
95% do ganho medio de precisao > 0,05, Wilcoxon pareado com correcao de Holm < 0,05, e taxa de
vitoria >= 70%, com no minimo 10 replicas pareadas.

In [2]:
REPLICATE_DIR = Path("datasets/synthetic_causal/replicates")
DATASETS = {
    "toy_a_linear": {
        "ground_truth_path": Path("datasets/synthetic_causal/toy_a_linear_gt.csv"),
        "replicate_dir": REPLICATE_DIR / "toy_a_linear",
    },
    "toy_b_nonlinear": {
        "ground_truth_path": Path("datasets/synthetic_causal/toy_b_nonlinear_gt.csv"),
        "replicate_dir": REPLICATE_DIR / "toy_b_nonlinear",
    },
}

RESULTS_DIR = Path(".local/results/toy_synthetic_validation_auto_selection")
CACHE_DIR = Path(".local/results/toy_synthetic_validation_auto_selection_cache")

ENSEMBLE_METHOD_NAMES = [
    "ClassicalGranger", "GES", "FCI", "DYNOTEARS",
    "LPCMCI", "NeuralGrangercMLP", "PCMCI", "VARLiNGAM",
]
FULL_COMBINATION_KEY = " + ".join(ENSEMBLE_METHOD_NAMES)
MIN_METHODS = 2
MAX_METHODS = 8
MAX_LAG = 2
ENSEMBLE_THRESHOLD = 0.50
N_BOOTSTRAP = 10
METHOD_CONSENSUS_THRESHOLD = 0.50
SOFT_VOTING_SUPPORT_THRESHOLD = 0.0
LOCAL_EXPERT_WEIGHT = 0.60
PREDICTIVE_VALIDATION_WEIGHT = 0.75
PREDICTIVE_VALIDATION_SPLITS = 3
PREDICTIVE_VALIDATION_RIDGE_ALPHA = 5.0
PREDICTIVE_VALIDATION_CONDITIONAL_PARENTS = 0
PREDICTIVE_VALIDATION_UNCERTAINTY_PENALTY = 0.50
PREDICTIVE_VALIDATION_RANK_EXPONENT = 0.35
RANKED_SELECTION_MAX_PAIR_DENSITY = 0.40
RANKED_SELECTION_RESCUE_BOOTSTRAP_MIN = 0.40
RANKED_SELECTION_RESCUE_PREDICTIVE_RANK_MIN = 0.60
RANKED_SELECTION_RESCUE_SUPPORT_MIN = 0.25
METHOD_REDUNDANCY_PENALTY = 0.20
RANDOM_STATE = 42
MAX_BOOTSTRAP_SECONDS = 900
PARALLEL_JOBS = max(1, min(4, (os.cpu_count() or 2) - 1))

MINIMUM_PRECISION_GAIN = 0.05
MINIMUM_WIN_RATE = 0.70
SIGNIFICANCE_LEVEL = 0.05
MIN_CONFIRMATORY_REPLICATES = 10
STATISTICAL_BOOTSTRAPS = 10_000

RETRY_FAILURES = True
STOP_ON_ERROR = False

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

for name, config in DATASETS.items():
    replicate_paths = sorted(config["replicate_dir"].glob("replicate_*.csv"))
    config["replicate_paths"] = replicate_paths
    print(f"{name}: {len(replicate_paths)} replicas encontradas em {config['replicate_dir']}")

print(f"\nMetodos do ensemble: {ENSEMBLE_METHOD_NAMES}")
print(f"MAX_LAG={MAX_LAG} | N_BOOTSTRAP={N_BOOTSTRAP} | RANDOM_STATE={RANDOM_STATE}")
print(f"CPUs detectadas: {os.cpu_count()} | PARALLEL_JOBS={PARALLEL_JOBS}")

toy_a_linear: 10 replicas encontradas em datasets\synthetic_causal\replicates\toy_a_linear
toy_b_nonlinear: 10 replicas encontradas em datasets\synthetic_causal\replicates\toy_b_nonlinear

Metodos do ensemble: ['ClassicalGranger', 'GES', 'FCI', 'DYNOTEARS', 'LPCMCI', 'NeuralGrangercMLP', 'PCMCI', 'VARLiNGAM']
MAX_LAG=2 | N_BOOTSTRAP=10 | RANDOM_STATE=42
CPUs detectadas: 16 | PARALLEL_JOBS=4


## 2. Ground truth e pre-processamento

O ground truth (mesmo CSV `Edge,Direct,Coefficient,Lag,Type` das versoes anteriores) e o mesmo
para todas as replicas de um dataset — so a estrutura causal importa, e ela nao muda entre
replicas. Cada replica e pre-processada individualmente com o mesmo `CausalPreprocessor`
(teste ADF + diferenciacao condicional + normalizacao).

In [3]:
nodes_by_dataset = {}
ground_truth_by_dataset = {}
truth_summary_by_dataset = {}

for name, config in DATASETS.items():
    probe_bundle = load_time_series_dataset(
        config["replicate_paths"][0], data_format="csv",
        ground_truth_path=config["ground_truth_path"], selected_columns=None,
    )
    nodes = list(probe_bundle.selected_columns)
    ground_truth = probe_bundle.ground_truth
    truth_summary = compute_undirected_skeleton_metrics(
        pd.DataFrame(columns=["source", "target", "lag"]), ground_truth, nodes=nodes,
    )
    nodes_by_dataset[name] = nodes
    ground_truth_by_dataset[name] = ground_truth
    truth_summary_by_dataset[name] = truth_summary
    print(f"--- {name} ---")
    print(f"Nos: {nodes}")
    print(f"Pares possiveis: {truth_summary['candidate_pairs']} | "
          f"Pares verdadeiros: {truth_summary['ground_truth_pairs']} | "
          f"Prevalencia: {truth_summary['ground_truth_prevalence']:.2%}")
    print(f"Arestas do ground truth: "
          f"{list(zip(ground_truth['source'], ground_truth['target']))}")


def load_and_preprocess_replicate(data_path, nodes):
    bundle = load_time_series_dataset(data_path, data_format="csv", selected_columns=nodes)
    preprocessor = CausalPreprocessor(
        bundle.data, significance_level=0.05, decomposition_period=None
    )
    processed = preprocessor.fit_transform(
        make_stationary=True, normalize=True, remove_trend=False, max_diffs=2,
    )
    return processed, preprocessor.differencing_orders

--- toy_a_linear ---
Nos: ['Y', 'X1', 'X3', 'X4', 'X0']
Pares possiveis: 10 | Pares verdadeiros: 3 | Prevalencia: 30.00%
Arestas do ground truth: [('X1', 'Y'), ('X3', 'Y'), ('X4', 'X1')]
--- toy_b_nonlinear ---
Nos: ['Y', 'X1', 'X3', 'X4', 'X0']
Pares possiveis: 10 | Pares verdadeiros: 3 | Prevalencia: 30.00%
Arestas do ground truth: [('X1', 'Y'), ('X3', 'Y'), ('X4', 'X1')]


## 3. Funcoes do experimento (identicas as usadas na versao 1, so os limiares mudaram)

In [4]:
def restrict_method_relations(method, allowed_relations):
    allowed_relations = set(allowed_relations)

    def run_restricted(data, **kwargs):
        result = method(data, **kwargs)
        if result is None or result.empty:
            return result
        mask = [
            (source, target) in allowed_relations
            for source, target in zip(result["source"], result["target"])
        ]
        return result.loc[mask].reset_index(drop=True)

    return run_restricted


def evidence_mode(frame):
    if "p_value" in frame.columns:
        p_values = pd.to_numeric(frame["p_value"], errors="coerce")
        if np.isfinite(p_values).any():
            return "one_minus_p_value"
    return "absolute_score"


def apply_precision_focused_consensus(summary):
    return add_precision_consensus_selection(
        summary,
        score_threshold=ENSEMBLE_THRESHOLD,
        method_support_threshold=METHOD_CONSENSUS_THRESHOLD,
    )


def apply_soft_voting_precision(summary):
    frame = summary.copy()
    frame["ensemble_score"] = frame["pre_validation_ensemble_score"]
    return add_precision_consensus_selection(
        frame,
        score_threshold=ENSEMBLE_THRESHOLD,
        method_support_threshold=SOFT_VOTING_SUPPORT_THRESHOLD,
    )


def evaluate_strategy(frame, strategy, nodes, ground_truth, runtime_seconds, *, probability=False):
    binary_frame = frame
    binary_threshold = ENSEMBLE_THRESHOLD
    if probability and "ensemble_selected" in frame:
        binary_frame = frame.loc[
            frame["ensemble_selected"].fillna(False).astype(bool)
        ].copy()
        binary_threshold = 0.0
    binary = compute_undirected_skeleton_metrics(
        binary_frame, ground_truth, prob_threshold=binary_threshold, nodes=nodes,
    )
    pair_scores = build_complete_undirected_pair_scores(
        frame, nodes,
        evidence=(
            "ensemble_score" if probability and "ensemble_score" in frame.columns
            else "probability" if probability
            else evidence_mode(frame)
        ),
    )
    ranked = compute_ranked_undirected_skeleton_metrics(pair_scores, ground_truth)
    return {
        "strategy": str(strategy),
        "precision": binary["precision"],
        "recall": binary["recall"],
        "f1_score": binary["f1_score"],
        "structural_hamming_distance": binary["structural_hamming_distance"],
        "true_positives": binary["true_positives"],
        "false_positives": binary["false_positives"],
        "false_negatives": binary["false_negatives"],
        "average_precision": ranked["average_precision"],
        "roc_auc": ranked["roc_auc"],
        "runtime_seconds": float(runtime_seconds),
    }


def baseline_rows(nodes, ground_truth, truth_summary, replicate_seed):
    pairs = [(nodes[i], nodes[j]) for i in range(len(nodes)) for j in range(i + 1, len(nodes))]
    all_pairs = pd.DataFrame([
        {"source": source, "target": target, "lag": 1, "score": 1.0, "p_value": np.nan}
        for source, target in pairs
    ])
    rng = np.random.default_rng(RANDOM_STATE + int(replicate_seed))
    random_scores = rng.random(len(pairs))
    true_pair_count = truth_summary["ground_truth_pairs"]
    selected = (
        np.argsort(random_scores)[-true_pair_count:]
        if true_pair_count else np.array([], dtype=int)
    )
    random_edges = pd.DataFrame([
        {"source": pairs[index][0], "target": pairs[index][1], "lag": 1,
         "score": random_scores[index], "p_value": np.nan}
        for index in selected
    ])
    random_pair_scores = pd.DataFrame([
        {"source": source, "target": target, "score": score}
        for (source, target), score in zip(pairs, random_scores)
    ])

    all_metrics = evaluate_strategy(all_pairs, "ALL_PAIRS", nodes, ground_truth, 0.0)
    random_binary = compute_undirected_skeleton_metrics(random_edges, ground_truth, nodes=nodes)
    random_ranked = compute_ranked_undirected_skeleton_metrics(random_pair_scores, ground_truth)
    random_metrics = {
        "strategy": "RANDOM_DENSITY",
        "precision": random_binary["precision"], "recall": random_binary["recall"],
        "f1_score": random_binary["f1_score"],
        "structural_hamming_distance": random_binary["structural_hamming_distance"],
        "true_positives": random_binary["true_positives"],
        "false_positives": random_binary["false_positives"],
        "false_negatives": random_binary["false_negatives"],
        "average_precision": random_ranked["average_precision"],
        "roc_auc": random_ranked["roc_auc"], "runtime_seconds": 0.0,
    }
    return [all_metrics, random_metrics]


def selection_arguments(processed_data):
    return {
        "min_methods": MIN_METHODS,
        "max_methods": MAX_METHODS,
        "min_votes": 1,
        "n_bootstrap": N_BOOTSTRAP,
        "block_size": max(2, len(processed_data) // 12),
        "stability_threshold": 0.60,
        "selection_probability_threshold": 0.50,
        "prior_edge_probability": 0.10,
        "posterior_weight": 0.70,
        "adaptive_method_weights": True,
        "stability_weight": 0.65,
        "local_expert_weight": LOCAL_EXPERT_WEIGHT,
        "predictive_validation_weight": PREDICTIVE_VALIDATION_WEIGHT,
        "predictive_validation_max_lag": MAX_LAG,
        "predictive_validation_splits": PREDICTIVE_VALIDATION_SPLITS,
        "predictive_validation_ridge_alpha": PREDICTIVE_VALIDATION_RIDGE_ALPHA,
        "predictive_validation_conditional_parents": PREDICTIVE_VALIDATION_CONDITIONAL_PARENTS,
        "predictive_validation_uncertainty_penalty": PREDICTIVE_VALIDATION_UNCERTAINTY_PENALTY,
        "predictive_validation_rank_exponent": PREDICTIVE_VALIDATION_RANK_EXPONENT,
        "ranked_selection_max_pair_density": RANKED_SELECTION_MAX_PAIR_DENSITY,
        "ranked_selection_rescue_bootstrap_min": RANKED_SELECTION_RESCUE_BOOTSTRAP_MIN,
        "ranked_selection_rescue_predictive_rank_min": RANKED_SELECTION_RESCUE_PREDICTIVE_RANK_MIN,
        "ranked_selection_rescue_support_min": RANKED_SELECTION_RESCUE_SUPPORT_MIN,
        "method_redundancy_penalty": METHOD_REDUNDANCY_PENALTY,
        "method_stability_power": 1.0,
        "method_diversity_bonus": 0.15,
        "method_density_penalty": 0.50,
        "minimum_method_weight": 0.05,
        "confidence_level": 0.95,
        "random_state": RANDOM_STATE,
        "precompute_runs": True,
        "parallel_jobs": PARALLEL_JOBS,
        "max_bootstrap_seconds": MAX_BOOTSTRAP_SECONDS,
    }

## 4. Execucao com checkpoint (10 replicas x 2 datasets)

Cada `(dataset, replica)` roda a busca combinatoria uma unica vez (`min_methods=2, max_methods=8`,
ate 247 subconjuntos avaliados) — o resultado ja contem tanto a avaliacao do subconjunto completo
de 8 metodos (`all_evaluations[FULL_COMBINATION_KEY]`, usada para a estrategia `ENSEMBLE`, igual
as versoes anteriores) quanto a avaliacao do melhor subconjunto encontrado
(`best_evaluation`/`best_combination`, usada para `ENSEMBLE_AUTO`) — nao e preciso rodar a suite
de metodos duas vezes. As saidas base e de bootstrap ficam em cache por pickle por
`(dataset, replica)`, e `metrics.csv`/`selections.csv`/`failures.csv` sao salvos apos cada replica
— a celula pode ser reexecutada com seguranca (pula o que ja esta pronto) se for interrompida.

In [5]:
all_registered_methods = get_registered_methods()
all_method_kwargs = get_registered_method_kwargs(MAX_LAG)
all_method_weights = get_registered_method_weights()
methods = {name: all_registered_methods[name] for name in ENSEMBLE_METHOD_NAMES}
method_kwargs = {name: all_method_kwargs[name] for name in ENSEMBLE_METHOD_NAMES}
method_weights = {name: all_method_weights[name] for name in ENSEMBLE_METHOD_NAMES}

metrics_path = RESULTS_DIR / "metrics.csv"
selections_path = RESULTS_DIR / "selections.csv"
failures_path = RESULTS_DIR / "failures.csv"

metrics_results = pd.read_csv(metrics_path) if metrics_path.exists() else pd.DataFrame()
selection_results = pd.read_csv(selections_path) if selections_path.exists() else pd.DataFrame()
failure_results = pd.read_csv(failures_path) if failures_path.exists() else pd.DataFrame()


def replicate_key(dataset_name, replicate_index):
    return f"{dataset_name}::{replicate_index}"


completed = set()
if not metrics_results.empty:
    ensemble_rows = metrics_results.loc[metrics_results["strategy"].eq("ENSEMBLE")]
    completed = {
        replicate_key(row.dataset, int(row.replicate_index))
        for row in ensemble_rows.itertuples()
    }
failed = set()
if not failure_results.empty:
    failed = {
        replicate_key(row.dataset, int(row.replicate_index))
        for row in failure_results.itertuples()
    }


def run_replicate(name, replicate_index, data_path, seed, nodes, ground_truth, truth_summary):
    processed, differencing_orders = load_and_preprocess_replicate(data_path, nodes)

    relations = {
        (source, target) for source in nodes for target in nodes if source != target
    }
    restricted = {
        method_name: restrict_method_relations(method, relations)
        for method_name, method in methods.items()
    }

    cache_path = CACHE_DIR / f"{name}_replicate{replicate_index:02d}.pkl"
    cache_signature = {
        "dataset": name, "replicate_index": replicate_index, "nodes": list(nodes),
        "processed_rows": len(processed), "max_lag": MAX_LAG, "n_bootstrap": N_BOOTSTRAP,
        "ensemble_methods": list(ENSEMBLE_METHOD_NAMES),
        "min_methods": MIN_METHODS, "max_methods": MAX_METHODS,
        "local_expert_weight": LOCAL_EXPERT_WEIGHT,
        "method_redundancy_penalty": METHOD_REDUNDANCY_PENALTY,
        "random_state": RANDOM_STATE,
    }
    cached_payload = None
    if cache_path.exists():
        with cache_path.open("rb") as stream:
            candidate_payload = pickle.load(stream)
        if candidate_payload.get("signature") == cache_signature:
            cached_payload = candidate_payload

    started = time.perf_counter()
    selection = select_robust_ensemble_combination(
        processed, restricted,
        method_kwargs=method_kwargs, method_weights=method_weights,
        expert_knowledge=[],
        precomputed_outputs=(cached_payload.get("outputs") if cached_payload else None),
        precomputed_bootstrap_outputs=(
            cached_payload.get("bootstrap_outputs") if cached_payload else None
        ),
        **selection_arguments(processed),
    )
    recalculation_runtime = time.perf_counter() - started
    cached_base_runtime = cached_payload.get("base_runtime_seconds") if cached_payload else None
    ensemble_runtime = (
        float(cached_base_runtime)
        if cached_base_runtime is not None and np.isfinite(cached_base_runtime)
        else recalculation_runtime
    )
    if cached_payload is None:
        with cache_path.open("wb") as stream:
            pickle.dump({
                "signature": cache_signature,
                "outputs": selection["precomputed_outputs"],
                "bootstrap_outputs": selection["precomputed_bootstrap_outputs"],
                "base_runtime_seconds": recalculation_runtime,
            }, stream, protocol=pickle.HIGHEST_PROTOCOL)

    individual_outputs = {}
    for evaluation in selection["all_evaluations"].values():
        for method_name, output in evaluation["outputs"].items():
            individual_outputs.setdefault(method_name, output)

    rows = [
        evaluate_strategy(
            individual_outputs[method_name], method_name, nodes, ground_truth, np.nan
        )
        for method_name in ENSEMBLE_METHOD_NAMES
    ]

    full8_evaluation = selection["all_evaluations"][FULL_COMBINATION_KEY]
    auto_evaluation = selection["best_evaluation"]

    summary_before_consensus = full8_evaluation["probabilistic_summary"].copy()
    summary_hard_consensus = apply_precision_focused_consensus(summary_before_consensus)
    summary_fixed8 = apply_soft_voting_precision(summary_before_consensus)
    summary_without_gate = summary_before_consensus.copy()
    summary_without_gate["ensemble_score"] = summary_without_gate["pre_validation_ensemble_score"]
    summary_without_gate = add_ranked_structure_selection(
        summary_without_gate, nodes=nodes,
        max_pair_density=RANKED_SELECTION_MAX_PAIR_DENSITY,
    )
    summary_auto = apply_soft_voting_precision(auto_evaluation["probabilistic_summary"].copy())

    rows.append(evaluate_strategy(
        summary_hard_consensus, "ENSEMBLE_MAIORIA_RIGIDA", nodes, ground_truth,
        ensemble_runtime, probability=True,
    ))
    rows.append(evaluate_strategy(
        summary_without_gate, "ENSEMBLE_SEM_GATE", nodes, ground_truth,
        ensemble_runtime, probability=True,
    ))
    rows.append(evaluate_strategy(
        summary_before_consensus, "ENSEMBLE_TOP_K_ANTERIOR", nodes, ground_truth,
        ensemble_runtime, probability=True,
    ))
    rows.append(evaluate_strategy(
        summary_fixed8, "ENSEMBLE", nodes, ground_truth, ensemble_runtime, probability=True,
    ))
    rows.append(evaluate_strategy(
        summary_auto, "ENSEMBLE_AUTO", nodes, ground_truth, ensemble_runtime, probability=True,
    ))
    rows.extend(baseline_rows(nodes, ground_truth, truth_summary, seed))
    for row in rows:
        row["dataset"] = name
        row["replicate_index"] = int(replicate_index)
        row["seed"] = int(seed)
        row["differencing_orders"] = json.dumps(differencing_orders)

    selected_pair_count_fixed8 = len({
        tuple(sorted((str(row.source), str(row.target))))
        for row in summary_fixed8.loc[summary_fixed8["ensemble_selected"]].itertuples()
    })
    selected_pair_count_auto = len({
        tuple(sorted((str(row.source), str(row.target))))
        for row in summary_auto.loc[summary_auto["ensemble_selected"]].itertuples()
    })
    ranking = selection["ranking"].set_index("combination")
    selection_row = {
        "dataset": name, "replicate_index": int(replicate_index), "seed": int(seed),
        "auto_combination": " + ".join(selection["best_combination"]),
        "auto_performance_score": float(selection["ranking"].iloc[0]["performance_score"]),
        "full8_performance_score": float(ranking.loc[FULL_COMBINATION_KEY, "performance_score"]),
        "combinations_evaluated": int(len(selection["ranking"])),
        "ensemble_runtime_seconds": ensemble_runtime,
        "cache_reused": cached_payload is not None,
        "recalculation_runtime_seconds": recalculation_runtime,
        "processed_rows": len(processed),
        "bootstrap_iterations_used": len(selection["precomputed_bootstrap_outputs"]),
        "selected_pair_count_fixed8": selected_pair_count_fixed8,
        "selected_pair_count_auto": selected_pair_count_auto,
        "auto_adaptive_weights": json.dumps(
            auto_evaluation["effective_method_weights"], ensure_ascii=False, sort_keys=True,
        ),
    }
    return rows, selection_row


replicate_jobs = [
    (name, index, path, int(path.stem.rsplit("seed", 1)[1]))
    for name, config in DATASETS.items()
    for index, path in enumerate(config["replicate_paths"], start=1)
]

for position, (name, replicate_index, data_path, seed) in enumerate(replicate_jobs, start=1):
    key = replicate_key(name, replicate_index)
    if key in completed or (key in failed and not RETRY_FAILURES):
        print(f"[{position}/{len(replicate_jobs)}] {name} replica {replicate_index}: checkpoint")
        continue
    print(f"[{position}/{len(replicate_jobs)}] {name} replica {replicate_index} (seed={seed}): executando...")
    started = time.perf_counter()
    try:
        rows, selection_row = run_replicate(
            name, replicate_index, data_path, seed,
            nodes_by_dataset[name], ground_truth_by_dataset[name], truth_summary_by_dataset[name],
        )
        metrics_results = pd.concat([metrics_results, pd.DataFrame(rows)], ignore_index=True)
        selection_results = pd.concat(
            [selection_results, pd.DataFrame([selection_row])], ignore_index=True
        )
        metrics_results.to_csv(metrics_path, index=False)
        selection_results.to_csv(selections_path, index=False)
        if not failure_results.empty:
            failure_results = failure_results.loc[
                ~failure_results.apply(
                    lambda row: replicate_key(row["dataset"], int(row["replicate_index"])) == key,
                    axis=1,
                )
            ].reset_index(drop=True)
            if failure_results.empty:
                failures_path.unlink(missing_ok=True)
            else:
                failure_results.to_csv(failures_path, index=False)
        print(f"  concluida em {(time.perf_counter() - started) / 60:.1f} min")
    except Exception as error:
        failure_row = {
            "dataset": name, "replicate_index": int(replicate_index),
            "error_type": type(error).__name__, "error": str(error),
        }
        failure_results = pd.concat(
            [failure_results, pd.DataFrame([failure_row])], ignore_index=True
        )
        failure_results.to_csv(failures_path, index=False)
        print(f"  FALHA: {type(error).__name__}: {error}")
        if STOP_ON_ERROR:
            raise

completed_count = (
    metrics_results.loc[metrics_results["strategy"].eq("ENSEMBLE")]
    .groupby("dataset")["replicate_index"].nunique()
    if not metrics_results.empty else pd.Series(dtype=int)
)
print(f"\nReplicas completas por dataset:\n{completed_count}")
print(f"Falhas registradas: {len(failure_results)}")

[1/20] toy_a_linear replica 1: checkpoint
[2/20] toy_a_linear replica 2 (seed=1002): executando...


  concluida em 7.5 min
[3/20] toy_a_linear replica 3 (seed=1003): executando...


  concluida em 7.2 min
[4/20] toy_a_linear replica 4 (seed=1004): executando...


  concluida em 7.4 min
[5/20] toy_a_linear replica 5 (seed=1005): executando...


  concluida em 7.2 min
[6/20] toy_a_linear replica 6 (seed=1006): executando...


...

  concluida em 6.7 min
[17/20] toy_b_nonlinear replica 7 (seed=1007): executando...


  concluida em 8.8 min
[18/20] toy_b_nonlinear replica 8 (seed=1008): executando...


  concluida em 7.1 min
[19/20] toy_b_nonlinear replica 9 (seed=1009): executando...


  concluida em 5.9 min
[20/20] toy_b_nonlinear replica 10 (seed=1010): executando...


  concluida em 6.8 min

Replicas completas por dataset:
dataset
toy_a_linear       10
toy_b_nonlinear    10
Name: replicate_index, dtype: int64
Falhas registradas: 0


## 5. Resultados descritivos (10 replicas por dataset)

Com 10 pontos por estrategia agora faz sentido olhar a distribuicao (boxplot), nao so a media.
Tambem mostramos quais subconjuntos de metodos a busca escolheu em cada replica — se ela convergir
quase sempre para os mesmos 2-3 metodos, isso e evidencia (dentro destes dois grafos) de que o
`performance_score` interno esta identificando os metodos realmente mais confiaveis.

In [6]:
main_strategies = [*ENSEMBLE_METHOD_NAMES, "ENSEMBLE", "ENSEMBLE_AUTO"]
strategy_order = ["ENSEMBLE_AUTO", "ENSEMBLE", *ENSEMBLE_METHOD_NAMES]

for name in DATASETS:
    subset = metrics_results[
        (metrics_results["dataset"] == name) & (metrics_results["strategy"].isin(main_strategies))
    ].copy()
    summary_table = subset.groupby("strategy")[[
        "precision", "recall", "f1_score", "average_precision", "roc_auc",
    ]].agg(["mean", "std"]).round(4)
    summary_table = summary_table.reindex(strategy_order)
    print(f"=== {name} — media e desvio-padrao entre as 10 replicas ===")
    display(summary_table)

    figure = px.box(
        subset, x="strategy", y="precision", points="all",
        category_orders={"strategy": strategy_order},
        title=f"Precisao por estrategia — {name} (10 replicas)",
    )
    figure.update_xaxes(tickangle=45)
    figure.show()

    combo_counts = selection_results.loc[
        selection_results["dataset"] == name, "auto_combination"
    ].value_counts()
    print(f"Subconjuntos escolhidos pela busca em {name} (10 replicas):")
    print(combo_counts.to_string())
    print()

=== toy_a_linear — media e desvio-padrao entre as 10 replicas ===


precision          recall         f1_score          \
                       mean     std    mean     std     mean     std   
strategy                                                               
ENSEMBLE_AUTO         1.000  0.0000  0.9667  0.1054   0.9800  0.0632   
ENSEMBLE              0.975  0.0791  1.0000  0.0000   0.9857  0.0452   
ClassicalGranger      0.665  0.0944  1.0000  0.0000   0.7952  0.0698   
GES                   0.935  0.1415  0.9667  0.1054   0.9407  0.0987   
FCI                   0.900  0.1291  1.0000  0.0000   0.9429  0.0738   
DYNOTEARS             1.000  0.0000  1.0000  0.0000   1.0000  0.0000   
LPCMCI                1.000  0.0000  0.9333  0.1405   0.9600  0.0843   
NeuralGrangercMLP     0.300  0.0000  1.0000  0.0000   0.4615  0.0000   
PCMCI                 0.740  0.1941  1.0000  0.0000   0.8381  0.1246   
VARLiNGAM             1.000  0.0000  1.0000  0.0000   1.0000  0.0000   

                  average_precision         roc_auc          
                               mean     std    mean     std  
strategy                                                     
ENSEMBLE_AUTO                1.0000  0.0000  1.0000  0.0000  
ENSEMBLE                     1.0000  0.0000  1.0000  0.0000  
ClassicalGranger             0.7500  0.0000  0.9286  0.0000  
GES                          0.9117  0.1487  0.9619  0.0656  
FCI                          0.9000  0.1291  0.9714  0.0369  
DYNOTEARS                    1.0000  0.0000  1.0000  0.0000  
LPCMCI                       0.9533  0.0984  0.9667  0.0703  
NeuralGrangercMLP            0.7847  0.0073  0.7381  0.0251  
PCMCI                        1.0000  0.0000  1.0000  0.0000  
VARLiNGAM                    1.0000  0.0000  1.0000  0.0000

Subconjuntos escolhidos pela busca em toy_a_linear (10 replicas):
auto_combination
DYNOTEARS + LPCMCI + VARLiNGAM                        2
GES + DYNOTEARS + LPCMCI + VARLiNGAM                  2
GES + FCI + DYNOTEARS + LPCMCI + PCMCI + VARLiNGAM    2
FCI + DYNOTEARS + VARLiNGAM                           1
DYNOTEARS + VARLiNGAM                                 1
GES + FCI + DYNOTEARS + LPCMCI + VARLiNGAM            1
FCI + DYNOTEARS + LPCMCI + VARLiNGAM                  1

=== toy_b_nonlinear — media e desvio-padrao entre as 10 replicas ===


precision          recall         f1_score          \
                       mean     std    mean     std     mean     std   
strategy                                                               
ENSEMBLE_AUTO        1.0000  0.0000  0.9000  0.1610   0.9400  0.0966   
ENSEMBLE             0.9750  0.0791  1.0000  0.0000   0.9857  0.0452   
ClassicalGranger     0.6500  0.0913  1.0000  0.0000   0.7845  0.0674   
GES                  1.0000  0.0000  0.9333  0.1405   0.9600  0.0843   
FCI                  0.9000  0.1291  1.0000  0.0000   0.9429  0.0738   
DYNOTEARS            1.0000  0.0000  1.0000  0.0000   1.0000  0.0000   
LPCMCI               0.9017  0.1622  0.8667  0.1721   0.8674  0.1239   
NeuralGrangercMLP    0.3000  0.0000  1.0000  0.0000   0.4615  0.0000   
PCMCI                0.7150  0.1717  1.0000  0.0000   0.8238  0.1115   
VARLiNGAM            1.0000  0.0000  1.0000  0.0000   1.0000  0.0000   

                  average_precision         roc_auc          
                               mean     std    mean     std  
strategy                                                     
ENSEMBLE_AUTO                1.0000  0.0000  1.0000  0.0000  
ENSEMBLE                     1.0000  0.0000  1.0000  0.0000  
ClassicalGranger             0.7500  0.0000  0.9286  0.0000  
GES                          0.9533  0.0984  0.9667  0.0703  
FCI                          0.9000  0.1291  0.9714  0.0369  
DYNOTEARS                    1.0000  0.0000  1.0000  0.0000  
LPCMCI                       0.9067  0.1205  0.9310  0.0894  
NeuralGrangercMLP            0.8095  0.0000  0.8095  0.0000  
PCMCI                        1.0000  0.0000  1.0000  0.0000  
VARLiNGAM                    1.0000  0.0000  1.0000  0.0000

Subconjuntos escolhidos pela busca em toy_b_nonlinear (10 replicas):
auto_combination
DYNOTEARS + VARLiNGAM                         3
GES + FCI + DYNOTEARS + LPCMCI + VARLiNGAM    2
GES + DYNOTEARS + VARLiNGAM                   2
GES + FCI + DYNOTEARS + VARLiNGAM             1
GES + DYNOTEARS + LPCMCI + VARLiNGAM          1
GES + FCI + LPCMCI                            1



## 6. Teste estatistico pareado (ENSEMBLE_AUTO vs cada metodo e vs ENSEMBLE fixo, por dataset)

Mesmo criterio do notebook Traffic, aplicado separadamente a cada grafo toy: o `ENSEMBLE_AUTO` e
considerado superior a um baseline se, simultaneamente, (1) o limite inferior do IC 95% do ganho
medio de precisao for maior que `0,05`; (2) o p-valor pareado de Wilcoxon, corrigido por Holm, for
menor que `0,05`; (3) vencer em pelo menos 70% das replicas pareadas; com no minimo 10 replicas
pareadas disponiveis. O candidato principal agora e `ENSEMBLE_AUTO` (nao mais `ENSEMBLE` fixo) —
comparado contra os 8 metodos avulsos e, adicionalmente, contra o proprio `ENSEMBLE` fixo, para
medir se a selecao automatica de fato ajudou.

In [7]:
def holm_adjust(p_values):
    values = np.asarray(p_values, dtype=float)
    order = np.argsort(values)
    adjusted_sorted = np.maximum.accumulate(
        np.array([(len(values) - rank) * values[index] for rank, index in enumerate(order)])
    )
    adjusted = np.empty_like(values)
    adjusted[order] = np.minimum(adjusted_sorted, 1.0)
    return adjusted


primary_comparisons = {}
for name in DATASETS:
    subset = metrics_results[metrics_results["dataset"] == name].rename(
        columns={"replicate_index": "trajectory_index"}
    )
    baselines = [*ENSEMBLE_METHOD_NAMES, "ENSEMBLE"]
    primary_rows = [
        compute_paired_superiority_statistics(
            subset, candidate="ENSEMBLE_AUTO", baseline=baseline, metric="precision",
            higher_is_better=True, n_bootstrap=STATISTICAL_BOOTSTRAPS, random_state=RANDOM_STATE,
        )
        for baseline in baselines
    ]
    primary_comparison = pd.DataFrame(primary_rows)
    primary_comparison["holm_p_value"] = holm_adjust(primary_comparison["wilcoxon_p_value"])
    primary_comparison["confirmatory_sample_available"] = (
        primary_comparison["paired_trajectories"] >= MIN_CONFIRMATORY_REPLICATES
    )
    primary_comparison["superiority_criterion_met"] = (
        primary_comparison["confirmatory_sample_available"]
    ) & (
        primary_comparison["confidence_interval_low"] > MINIMUM_PRECISION_GAIN
    ) & (
        primary_comparison["holm_p_value"] < SIGNIFICANCE_LEVEL
    ) & (
        primary_comparison["win_rate"] >= MINIMUM_WIN_RATE
    )
    primary_comparisons[name] = primary_comparison
    print(f"=== {name} — ENSEMBLE_AUTO vs cada metodo e vs ENSEMBLE fixo (metrica: precisao) ===")
    display(primary_comparison.sort_values("baseline").round(4))

=== toy_a_linear — ENSEMBLE_AUTO vs cada metodo e vs ENSEMBLE fixo (metrica: precisao) ===


,candidate,baseline,metric,paired_trajectories,candidate_mean,baseline_mean,mean_improvement,median_improvement,confidence_interval_low,confidence_interval_high,win_rate,tie_rate,standardized_effect,wilcoxon_p_value,holm_p_value,confirmatory_sample_available,superiority_criterion_met
0,ENSEMBLE_AUTO,ClassicalGranger,precision,10,1.0,0.665,0.335,0.325,0.280,0.390,1.0,0.0,3.5477,0.0020,0.0176,True,True
3,ENSEMBLE_AUTO,DYNOTEARS,precision,10,1.0,1.000,0.000,0.000,0.000,0.000,0.0,1.0,0.0000,1.0000,1.0000,True,False
8,ENSEMBLE_AUTO,ENSEMBLE,precision,10,1.0,0.975,0.025,0.000,0.000,0.075,0.1,0.9,0.3162,1.0000,1.0000,True,False
2,ENSEMBLE_AUTO,FCI,precision,10,1.0,0.900,0.100,0.000,0.025,0.175,0.4,0.6,0.7746,0.1250,0.7500,True,False
1,ENSEMBLE_AUTO,GES,precision,10,1.0,0.935,0.065,0.000,0.000,0.155,0.2,0.8,0.4593,0.5000,1.0000,True,False
4,ENSEMBLE_AUTO,LPCMCI,precision,10,1.0,1.000,0.000,0.000,0.000,0.000,0.0,1.0,0.0000,1.0000,1.0000,True,False
5,ENSEMBLE_AUTO,NeuralGrangercMLP,precision,10,1.0,0.300,0.700,0.700,0.700,0.700,1.0,0.0,inf,0.0020,0.0176,True,True
6,ENSEMBLE_AUTO,PCMCI,precision,10,1.0,0.740,0.260,0.325,0.145,0.370,0.7,0.3,1.3397,0.0156,0.1094,True,False
7,ENSEMBLE_AUTO,VARLiNGAM,precision,10,1.0,1.000,0.000,0.000,0.000,0.000,0.0,1.0,0.0000,1.0000,1.0000,True,False


=== toy_b_nonlinear — ENSEMBLE_AUTO vs cada metodo e vs ENSEMBLE fixo (metrica: precisao) ===


,candidate,baseline,metric,paired_trajectories,candidate_mean,baseline_mean,mean_improvement,median_improvement,confidence_interval_low,confidence_interval_high,win_rate,tie_rate,standardized_effect,wilcoxon_p_value,holm_p_value,confirmatory_sample_available,superiority_criterion_met
0,ENSEMBLE_AUTO,ClassicalGranger,precision,10,1.0,0.6500,0.3500,0.400,0.295,0.4050,1.0,0.0,3.8341,0.0020,0.0176,True,True
3,ENSEMBLE_AUTO,DYNOTEARS,precision,10,1.0,1.0000,0.0000,0.000,0.000,0.0000,0.0,1.0,0.0000,1.0000,1.0000,True,False
8,ENSEMBLE_AUTO,ENSEMBLE,precision,10,1.0,0.9750,0.0250,0.000,0.000,0.0750,0.1,0.9,0.3162,1.0000,1.0000,True,False
2,ENSEMBLE_AUTO,FCI,precision,10,1.0,0.9000,0.1000,0.000,0.025,0.1750,0.4,0.6,0.7746,0.1250,0.7500,True,False
1,ENSEMBLE_AUTO,GES,precision,10,1.0,1.0000,0.0000,0.000,0.000,0.0000,0.0,1.0,0.0000,1.0000,1.0000,True,False
4,ENSEMBLE_AUTO,LPCMCI,precision,10,1.0,0.9017,0.0983,0.000,0.000,0.1983,0.3,0.7,0.6061,0.2500,1.0000,True,False
5,ENSEMBLE_AUTO,NeuralGrangercMLP,precision,10,1.0,0.3000,0.7000,0.700,0.700,0.7000,1.0,0.0,inf,0.0020,0.0176,True,True
6,ENSEMBLE_AUTO,PCMCI,precision,10,1.0,0.7150,0.2850,0.325,0.180,0.3800,0.8,0.2,1.6601,0.0078,0.0547,True,False
7,ENSEMBLE_AUTO,VARLiNGAM,precision,10,1.0,1.0000,0.0000,0.000,0.000,0.0000,0.0,1.0,0.0000,1.0000,1.0000,True,False


## 7. Conclusao automatica

Aplica o mesmo criterio confirmatorio do notebook Traffic a cada grafo toy separadamente.

In [8]:
for name, primary_comparison in primary_comparisons.items():
    total_baselines = len(primary_comparison)
    passed = primary_comparison.loc[primary_comparison["superiority_criterion_met"], "baseline"].tolist()
    failed_baselines = primary_comparison.loc[~primary_comparison["superiority_criterion_met"], "baseline"].tolist()
    confirmatory_available = bool(primary_comparison["confirmatory_sample_available"].all())

    print(f"=== {name} ===")
    if not confirmatory_available:
        print(
            f"Somente {int(primary_comparison['paired_trajectories'].min())} replicas pareadas; "
            f"minimo operacional definido e {MIN_CONFIRMATORY_REPLICATES}."
        )
        print(
            "Conclusao permitida: os resultados sugerem ou nao sugerem vantagem do "
            "ENSEMBLE_AUTO neste piloto. Reporte ganho medio, intervalo, taxa de vitorias e "
            "custo, sem afirmar superioridade estatisticamente demonstrada."
        )
    elif len(passed) == total_baselines:
        print(
            f"Conclusao permitida: no conjunto de {MIN_CONFIRMATORY_REPLICATES} replicas "
            f"independentes do grafo {name}, o ENSEMBLE_AUTO (selecao automatica de "
            "subconjunto) apresentou precisao superior a todos os 8 algoritmos avulsos e ao "
            "ENSEMBLE fixo de 8 metodos, segundo o criterio pre-especificado."
        )
    else:
        print(
            "Conclusao permitida: nao foi demonstrada superioridade do ENSEMBLE_AUTO sobre "
            "todos os baselines segundo o criterio pre-especificado. Reporte quais "
            "comparacoes foram positivas e quais permaneceram inconclusivas."
        )
    print("Comparacoes sem criterio confirmatorio atendido:", failed_baselines)
    print(
        f"Nao concluir superioridade universal sem repetir o protocolo em datasets com grafos "
        f"diferentes de {name}.\n"
    )

=== toy_a_linear ===
Conclusao permitida: nao foi demonstrada superioridade do ENSEMBLE_AUTO sobre todos os baselines segundo o criterio pre-especificado. Reporte quais comparacoes foram positivas e quais permaneceram inconclusivas.
Comparacoes sem criterio confirmatorio atendido: ['GES', 'FCI', 'DYNOTEARS', 'LPCMCI', 'PCMCI', 'VARLiNGAM', 'ENSEMBLE']
Nao concluir superioridade universal sem repetir o protocolo em datasets com grafos diferentes de toy_a_linear.

=== toy_b_nonlinear ===
Conclusao permitida: nao foi demonstrada superioridade do ENSEMBLE_AUTO sobre todos os baselines segundo o criterio pre-especificado. Reporte quais comparacoes foram positivas e quais permaneceram inconclusivas.
Comparacoes sem criterio confirmatorio atendido: ['GES', 'FCI', 'DYNOTEARS', 'LPCMCI', 'PCMCI', 'VARLiNGAM', 'ENSEMBLE']
Nao concluir superioridade universal sem repetir o protocolo em datasets com grafos diferentes de toy_b_nonlinear.



## 8. Conclusao e limites

Este notebook ativa a busca combinatoria de subconjuntos ja existente no framework
(`min_methods=2, max_methods=8`), guiada por um `performance_score` interno calculado so a partir
de sinais cegos (estabilidade de bootstrap, confianca, densidade) — em nenhum momento o ground
truth entra na selecao de metodos ou de arestas: ele so aparece nas celulas de avaliacao, depois
que ENSEMBLE, ENSEMBLE_AUTO e os 8 metodos avulsos ja produziram suas previsoes. O mesmo teste
estatistico pareado (Wilcoxon + Holm + IC 95% + taxa de vitoria, 10 replicas independentes por
grafo) da versao anterior e mantido, agora comparando `ENSEMBLE_AUTO` contra os 8 metodos avulsos
e contra o `ENSEMBLE` fixo de 8.

Ainda assim, a conclusao continua restrita a estes dois grafos toy especificos (mesma limitacao
que o Traffic tem em relacao ao seu proprio grafo) — nao e uma prova de que a selecao automatica
supera a combinacao fixa de 8 metodos, ou qualquer algoritmo avulso, em qualquer estrutura causal.